# Fine-tune All Seven-Emotion Models (GoEmotions + YouTube-Domain)

This notebook handles the complete training pipeline:

**Part A — Stage 1:** Fine-tune `distilbert-base-uncased` on balanced GoEmotions seven-class data.

**Part B — Stage 2:** Continue training on YouTube-domain DeepSeek-labeled data with class-weighted loss.

**Part C — Multi-model:** Domain-adapt other benchmark models (SamLowe RoBERTa, j-hartmann DistilRoBERTa, j-hartmann RoBERTa-large) on the same YouTube-domain split.

**Part D — Upload:** Push all adapted models to Hugging Face.

Labels: `anger`, `disgust`, `fear`, `joy`, `neutral`, `sadness`, `surprise`

Recommended runtime: **T4 GPU or better**.

## 1. Install dependencies

In [ ]:
!pip install -q "transformers>=4.41,<5.0.0" "datasets>=2.20" "accelerate>=0.30" "evaluate>=0.4" "scikit-learn>=1.5" "pandas>=2.2" "pyarrow>=15.0" "torch>=2.2" huggingface_hub

## 2. Clone or update the project repository

In [ ]:
import os
from pathlib import Path

repo_url = "https://github.com/chasezhang1999/youtube-emotion-analyzer.git"
repo_dir = Path("/content/youtube-emotion-analyzer")

if repo_dir.exists():
    %cd /content/youtube-emotion-analyzer
    !git pull
else:
    %cd /content
    !git clone {repo_url}
    %cd /content/youtube-emotion-analyzer

print("Working directory:", Path.cwd())

## 3. Imports and setup

In [ ]:
import inspect
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    pipeline,
)

import datasets.config as datasets_config
datasets_config.TORCHVISION_AVAILABLE = False

SEED = 5240
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TARGET_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
LABEL_TO_ID = {label: i for i, label in enumerate(TARGET_LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

# GitHub raw URLs for YouTube-domain data (used in Part B)
GITHUB_RAW = "https://raw.githubusercontent.com/chasezhang1999/youtube-emotion-analyzer/main"
YT_TRAIN_URL = f"{GITHUB_RAW}/data/youtube_domain_7class_deepseek/train.csv"
YT_VAL_URL = f"{GITHUB_RAW}/data/youtube_domain_7class_deepseek/validation.csv"

print("Labels:", TARGET_LABELS)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Helper functions

In [ ]:
def make_training_args(**kwargs):
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    else:
        kwargs["evaluation_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    return TrainingArguments(**kwargs)


def make_trainer(model, args, train_dataset, eval_dataset, tokenizer, compute_metrics_fn, class_weights=None, early_stopping_patience=2):
    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "compute_metrics": compute_metrics_fn,
    }
    sig = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in sig:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig:
        kwargs["tokenizer"] = tokenizer

    if early_stopping_patience > 0:
        kwargs["callbacks"] = [EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)]

    if class_weights is not None:
        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
        if torch.cuda.is_available():
            class_weights_tensor = class_weights_tensor.cuda()

        class WeightedTrainer(Trainer):
            def compute_loss(self, model, inputs, return_outputs=False, **kw):
                labels = inputs.pop("labels")
                outputs = model(**inputs)
                logits = outputs.logits
                loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
                loss = loss_fn(logits, labels)
                return (loss, outputs) if return_outputs else loss

        return WeightedTrainer(**kwargs)

    return Trainer(**kwargs)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    wp, wr, wf1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    mp, mr, mf1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "weighted_f1": wf1, "macro_f1": mf1,
            "weighted_precision": wp, "weighted_recall": wr,
            "macro_precision": mp, "macro_recall": mr}


def compute_class_weights_from_df(df):
    counts = df["label"].value_counts().sort_index().values.astype(float)
    weights = counts.sum() / (len(TARGET_LABELS) * counts)
    weights = weights / weights.min()
    return weights.tolist()


print("Helper functions loaded.")

---
## Part A — Stage 1: GoEmotions Fine-tuning

Train DistilBERT from scratch on balanced GoEmotions seven-class data.

## 5. Load GoEmotions dataset

Keep single-label samples for our 7 target emotions, then balance by downsampling.

In [ ]:
PARQUET_URLS = {
    "train": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
}

def filter_single_target(df):
    label_cols = [c for c in df.columns if c != "text"]
    single = df[df[label_cols].sum(axis=1) == 1].copy()
    target = single[single[TARGET_LABELS].sum(axis=1) == 1].copy()
    target["label_name"] = target[TARGET_LABELS].idxmax(axis=1)
    target["label"] = target["label_name"].map(LABEL_TO_ID).astype(int)
    return target[["text", "label_name", "label"]].reset_index(drop=True)

go_frames = {}
for split, url in PARQUET_URLS.items():
    raw = pd.read_parquet(url)
    filtered = filter_single_target(raw)
    go_frames[split] = filtered
    print(split, go_frames[split].shape)
    print(go_frames[split]["label_name"].value_counts().sort_index())
    print()

## 6. Convert to HF Dataset and tokenize

In [ ]:
go_dataset = DatasetDict({
    s: Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    for s, df in go_frames.items()
})

BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

go_tokenized = go_dataset.map(tokenize_batch, batched=True)
go_tokenized = go_tokenized.remove_columns(["text"])
go_tokenized

## 7. Stage 1: Train DistilBERT on GoEmotions

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(TARGET_LABELS), id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
)

s1_args = make_training_args(
    output_dir="./s1-goemotions-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

s1_trainer = make_trainer(model, s1_args, go_tokenized["train"], go_tokenized["validation"], tokenizer, compute_metrics, early_stopping_patience=0)

t0 = time.time()
s1_trainer.train()
print(f"\nStage 1 training time: {round(time.time() - t0, 2)}s")

## 8. Evaluate Stage 1

In [ ]:
s1_val = s1_trainer.evaluate(go_tokenized["validation"])
s1_test = s1_trainer.evaluate(go_tokenized["test"])
print("Stage 1 validation:", {k: round(v, 4) for k, v in s1_val.items()})
print("Stage 1 test:      ", {k: round(v, 4) for k, v in s1_test.items()})

## 9. Save Stage 1 model

In [ ]:
S1_DIR = "./youtube-emotion-distilbert"
s1_trainer.save_model(S1_DIR)
tokenizer.save_pretrained(S1_DIR)
print("Saved to", S1_DIR)

samples = [
    "I love this video so much!",
    "This campaign makes me angry.",
    "I am shocked by this announcement.",
    "This is just a normal update.",
]
pipe1 = pipeline("text-classification", model=S1_DIR, tokenizer=S1_DIR)
pipe1(samples, truncation=True, return_token_type_ids=False)

---
## Part B — Stage 2: YouTube-Domain Adaptation (DistilBERT)

Continue training the Stage 1 model on YouTube-domain DeepSeek-labeled data with class-weighted loss.

## 10. Load YouTube-domain DeepSeek data

The training data was pre-built by `scripts/build_deepseek_training_data.py` from 8,000 DeepSeek AI-labeled YouTube comments.

In [ ]:
yt_train = pd.read_csv(YT_TRAIN_URL)
yt_val = pd.read_csv(YT_VAL_URL)

# Ensure columns match expected format
for df in [yt_train, yt_val]:
    df["text"] = df["text"].astype(str)
    if "label_id" in df.columns:
        df["label"] = df["label_id"].astype(int)
    if "label_name" not in df.columns and "label" in df.columns:
        df["label_name"] = df["label"].map(ID_TO_LABEL) if df["label"].dtype == int else df["label"]

print(f"YouTube-domain train: {len(yt_train)}")
print(yt_train["label_name"].value_counts().sort_index())
print(f"\nYouTube-domain validation: {len(yt_val)}")
print(yt_val["label_name"].value_counts().sort_index())

## 11. Tokenize YouTube-domain data and compute class weights

In [ ]:
yt_dataset = DatasetDict({
    "train": Dataset.from_pandas(yt_train[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(yt_val[["text", "label"]], preserve_index=False),
})
yt_tokenized = yt_dataset.map(tokenize_batch, batched=True)
yt_tokenized = yt_tokenized.remove_columns(["text"])

class_weights = compute_class_weights_from_df(yt_train)
print("Class weights:")
for label, w in zip(TARGET_LABELS, class_weights):
    print(f"  {label}: {w:.2f}")

## 12. Stage 2: Domain adaptation training

In [ ]:
domain_model = AutoModelForSequenceClassification.from_pretrained(
    S1_DIR, num_labels=len(TARGET_LABELS), id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
)

s2_args = make_training_args(
    output_dir="./s2-domain-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

s2_trainer = make_trainer(
    domain_model, s2_args,
    yt_tokenized["train"], yt_tokenized["validation"],
    tokenizer, compute_metrics,
    class_weights=class_weights,
    early_stopping_patience=2,
)

t0 = time.time()
s2_trainer.train()
print(f"\nStage 2 training time: {round(time.time() - t0, 2)}s")

## 13. Evaluate and save Stage 2

In [ ]:
s2_val = s2_trainer.evaluate(yt_tokenized["validation"])
print("Stage 2 YouTube-domain validation:", {k: round(v, 4) for k, v in s2_val.items()})

S2_DIR = "./youtube-emotion-distilbert-domain-adapted"
s2_trainer.save_model(S2_DIR)
tokenizer.save_pretrained(S2_DIR)
print("Saved to", S2_DIR)

## 14. Compare Stage 1 vs Stage 2

In [ ]:
comparison = [
    "This launch is amazing and I want to buy it now!",
    "The brand response is terrible and people are angry.",
    "This safety ad is scary but important.",
    "I did not expect that ending at all.",
    "Just here to check the product details.",
    "This company should be ashamed of themselves.",
    "Watching from Ghana, love this!",
    "This made me cry so much.",
]

p1 = pipeline("text-classification", model=S1_DIR, tokenizer=S1_DIR)
p2 = pipeline("text-classification", model=S2_DIR, tokenizer=S2_DIR)

rows = []
for t in comparison:
    r1 = p1(t, truncation=True, return_token_type_ids=False)[0]
    r2 = p2(t, truncation=True, return_token_type_ids=False)[0]
    rows.append({"text": t, "stage1": r1["label"], "s1_conf": round(r1["score"], 3),
                 "domain": r2["label"], "dom_conf": round(r2["score"], 3)})
pd.DataFrame(rows)

---
## Part C — Multi-model Domain Adaptation

Train additional benchmark models on the same YouTube-domain split for fair comparison.
Uses `scripts/train_youtube_domain_all_models.py` which handles class weights, early stopping, and per-class metrics.

Models:
- `SamLowe/roberta-base-go_emotions` (public GoEmotions RoBERTa)
- `j-hartmann/emotion-english-distilroberta-base` (public 7-emotion DistilRoBERTa)
- `j-hartmann/emotion-english-roberta-large` (larger RoBERTa, optional — needs more VRAM)

## 15. Log in to Hugging Face

Add your HF write token to Colab Secrets as `HF_TOKEN` before running.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## 16. Dry-run training configuration

In [ ]:
!python scripts/train_youtube_domain_all_models.py --model all --hub-namespace chase1zhang --dry-run --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv

## 17. Train smaller / medium models

These three models are the most important fair-comparison set. Each uses the same YouTube-domain split.

In [ ]:
!python scripts/train_youtube_domain_all_models.py --model distilbert --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv --fp16
!python scripts/train_youtube_domain_all_models.py --model samlowe_roberta --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv --fp16
!python scripts/train_youtube_domain_all_models.py --model jhartmann_distilroberta --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv --fp16

## 18. Optional: train RoBERTa-large

Run only if Colab runtime and memory are sufficient. Uses smaller batch + gradient accumulation for T4.

In [ ]:
!python scripts/train_youtube_domain_all_models.py --model jhartmann_roberta_large --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 4 --gradient-accumulation-steps 2 --max-length 128 --train-csv data/youtube_domain_7class_deepseek/train.csv --validation-csv data/youtube_domain_7class_deepseek/validation.csv --fp16

## 19. Inspect all training metrics

In [ ]:
metrics_paths = sorted(Path("fine_tuned_model_files/youtube_domain_all_models").glob("*/youtube_domain_training_metrics.json"))
print(f"Found {len(metrics_paths)} metric files:\n")
for path in metrics_paths:
    data = json.loads(path.read_text())
    metrics = data.get("metrics", {})
    print(f"{path.parent.name}")
    print(f"  repo:       {data.get('repo_id')}")
    print(f"  accuracy:   {round(metrics.get('eval_accuracy', 0), 4)}")
    print(f"  macro_f1:   {round(metrics.get('eval_macro_f1', 0), 4)}")
    print(f"  elapsed:    {data.get('elapsed_seconds')}s")
    print()

---
## Part D — Upload DistilBERT models to Hugging Face

The multi-model training script (Part C) already pushes its own models. This section uploads the DistilBERT Stage 1 and Stage 2 models from Parts A/B.

In [ ]:
repo1 = "chase1zhang/youtube-emotion-distilbert"
repo2 = "chase1zhang/youtube-emotion-distilbert-domain-adapted"

s1_trainer.model.push_to_hub(repo1)
tokenizer.push_to_hub(repo1)
print(f"Uploaded Stage 1: https://huggingface.co/{repo1}")

s2_trainer.model.push_to_hub(repo2)
tokenizer.push_to_hub(repo2)
print(f"Uploaded domain-adapted: https://huggingface.co/{repo2}")

## 20. Verify uploaded models

In [ ]:
v1 = pipeline("text-classification", model=repo1, tokenizer=repo1)
v2 = pipeline("text-classification", model=repo2, tokenizer=repo2)
print("Stage 1:", v1(samples, truncation=True, return_token_type_ids=False))
print("Domain: ", v2(samples, truncation=True, return_token_type_ids=False))

## 21. Summary

Copy these into the report:
- Stage 1 GoEmotions validation / test metrics (cell 8)
- Stage 2 YouTube-domain validation metrics (cell 13)
- Multi-model domain adaptation metrics (cell 19)
- Hugging Face model URLs

The Streamlit app default model: `chase1zhang/youtube-emotion-distilbert-domain-adapted`